# CodeReview-Env — GRPO Training (Colab T4)

Train a code review agent using **Unsloth + HF TRL GRPOTrainer**.
Uses the live CodeReview-Env on HuggingFace Spaces for reward scoring.

### Prerequisites
- Runtime set to **T4 GPU** (Runtime > Change runtime type)
- Your HuggingFace Space must be running

In [ ]:
# Cell 1: Install Dependencies (Optimized for Colab T4)
%%capture
import os
os.environ["UNSLOTH_VLLM_STANDBY"] = "1"

# Install Unsloth from source (pulls compatible transformers)
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" -q

# Install TRL without heavy optional deps
!pip install --no-deps trl -q

# Install only what we actually need
!pip install peft accelerate bitsandbytes datasets requests matplotlib -q

# Remove packages that cause crashes (we don't use them)
!pip uninstall -y -q mergekit llm-blender vllm 2>/dev/null

In [ ]:
# Cell 2: Verify Installation
import torch
from unsloth import FastLanguageModel

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"✓ Unsloth loaded! GPU: {gpu} ({vram:.1f} GB VRAM)")
else:
    print("❌ No GPU! Go to Runtime > Change runtime type > T4 GPU")

In [ ]:
# Cell 3: Configuration
import os

# Environment API (your HuggingFace Space)
os.environ["ENV_URL"] = "https://dharaneswarreddy-codereview-env.hf.space"

# Groq API Key (paste your key from https://console.groq.com/keys)
os.environ["GROQ_API_KEY"] = "gsk_put_your_real_groq_key_here"

# HuggingFace Token (for pushing model to Hub, get from https://huggingface.co/settings/tokens)
os.environ["HF_TOKEN"] = "hf_put_your_real_hf_token_here"

# Model to train (Student)
os.environ["TRAIN_MODEL"] = "Qwen/Qwen2.5-1.5B-Instruct"

# Where to save the trained model on HuggingFace
os.environ["SAVE_REPO"] = "DharaneswarReddy/codereview-agent"

print("✓ Config set:")
print(f"  ENV_URL:     {os.environ.get('ENV_URL', '')}")
print(f"  TRAIN_MODEL: {os.environ.get('TRAIN_MODEL', '')}")
print(f"  SAVE_REPO:   {os.environ.get('SAVE_REPO', '')}")


In [ ]:
# Cell 4: Clone Repository & Start Training
!git clone https://github.com/GojoV339/MetaXScaler_Hackathon.git codereview-env 2>/dev/null || true
%cd codereview-env

# Run the GRPO training script
exec(open('training/train_grpo.py').read())

In [ ]:
# Cell 5: View Training Reward Curve
from IPython.display import Image, display
display(Image("reward_curve.png"))
print("\n↑ This shows how the agent's code review quality improved over training steps.")

In [ ]:
# Cell 3: Configuration
import os

# Environment API (your HuggingFace Space)
os.environ["ENV_URL"] = "https://dharaneswarreddy-codereview-env.hf.space"

# Groq API Key (paste your key from https://console.groq.com/keys)
os.environ["GROQ_API_KEY"] = "gsk_put_your_real_groq_key_here"

# HuggingFace Token (for pushing model to Hub, get from https://huggingface.co/settings/tokens)
os.environ["HF_TOKEN"] = "hf_put_your_real_hf_token_here"

# Model to train (Student)
os.environ["TRAIN_MODEL"] = "Qwen/Qwen2.5-1.5B-Instruct"

# Where to save the trained model on HuggingFace
os.environ["SAVE_REPO"] = "DharaneswarReddy/codereview-agent"

print("✓ Config set:")
    print(f"  ENV_URL:     {os.environ.get(\ENV_URL\, \)}")
    print(f"  TRAIN_MODEL: {os.environ.get(\TRAIN_MODEL\, \)}")
    print(f"  SAVE_REPO:   {os.environ.get(\SAVE_REPO\, \)}")


In [ ]:
# Cell 7: Compare Trained Model vs GPT-OSS 120B Baseline (Optional)
from openai import OpenAI

groq_client = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=os.environ["GROQ_API_KEY"],
)

test_obs = requests.post(f"{ENV_URL}/reset", json={"task_level": 1}).json()
groq_resp = groq_client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[
        {"role": "system", "content": "You are a senior code reviewer. Return valid JSON."},
        {"role": "user", "content": f"Review this code:\n```\n{test_obs['code']}\n```\nReturn JSON: {{\"has_bug\": bool, \"bug_type\": str, \"severity\": str, \"suggested_fix\": str}}"}
    ],
    max_tokens=200,
    temperature=0.6,
)
print(f"GPT-OSS 120B response:\n{groq_resp.choices[0].message.content}")